In [4]:
!nvidia-smi
!python --version
!nvcc --version

Sun Aug  2 21:01:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from pathlib import Path
import os
import subprocess


REPO_URL = (
    "https://github.com/"
    "Vihhycherezass/RAG-Labor-Code-RU.git"
)

BRANCH = "refactor/modular-rag"

REPO_DIR = Path(
    "/content/RAG-Labor-Code-RU"
)


if (REPO_DIR / ".git").exists():
    subprocess.run(
        [
            "git",
            "-C",
            str(REPO_DIR),
            "checkout",
            BRANCH,
        ],
        check=True,
    )

    subprocess.run(
        [
            "git",
            "-C",
            str(REPO_DIR),
            "pull",
            "--ff-only",
        ],
        check=True,
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )


os.chdir(REPO_DIR)

subprocess.run(
    [
        "git",
        "branch",
        "--show-current",
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "log",
        "-1",
        "--oneline",
    ],
    check=True,
)

print("Рабочая директория:", Path.cwd())

Рабочая директория: /content/RAG-Labor-Code-RU


In [6]:
%pip install "setuptools<82" jedi

%pip install \
    --force-reinstall \
    --no-cache-dir \
    --only-binary=:all: \
    "numpy==2.5.1" \
    "scipy==1.18.0"

%pip install \
    "llama-cpp-python==0.3.34" \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125 \
    --only-binary=:all: \
    --no-cache-dir

%pip install -e ".[dev]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 108.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 245.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 266.8 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.


Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu125
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 27.8 MB/s eta 0:00:0000:01m0:02m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 304.3 MB/s eta 0:00:00
Obtaining file:///content/RAG-Labor-Code-RU
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.8 MB/s eta 0:00:00
  Using cached setuptools-83.0.0-py3-none-any.whl.metadata (6.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 113.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 891.4/891.4 kB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.3/683.3 kB 56.9 MB/s eta 0:00:00
 

In [1]:
from importlib.metadata import version
from pathlib import Path
import os

import numpy
import scipy
import torch
import llama_cpp

from llama_cpp import Llama
from llama_index.retrievers.bm25 import BM25Retriever

from rag_labor_code.config import AppConfig
from rag_labor_code.bootstrap import build_rag_pipeline


REPO_DIR = Path(
    "/content/RAG-Labor-Code-RU"
)

os.chdir(REPO_DIR)


print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)

print(
    "llama-cpp-python:",
    version("llama-cpp-python"),
)

print(
    "rag-labor-code:",
    version("rag-labor-code"),
)

print(
    "CUDA доступна:",
    torch.cuda.is_available(),
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

print(
    "llama.cpp GPU offload:",
    llama_cpp.llama_supports_gpu_offload(),
)


assert torch.cuda.is_available(), (
    "PyTorch не видит CUDA."
)

assert (
    llama_cpp.llama_supports_gpu_offload()
), "llama.cpp установлен без CUDA."


print("Окружение полностью готово.")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


NumPy: 2.5.1
SciPy: 1.18.0
llama-cpp-python: 0.3.34
rag-labor-code: 0.1.0
CUDA доступна: True
GPU: Tesla T4
llama.cpp GPU offload: True
Окружение полностью готово.


In [2]:
!pytest -q

........................................................................ [ 18%]
........................................................................ [ 37%]
........................................................................ [ 55%]
........................................................................ [ 74%]
........................................................................ [ 93%]
...........................                                              [100%]
387 passed in 22.40s


In [3]:
from pathlib import Path

from huggingface_hub import hf_hub_download


MODELS_DIR = Path(
    "/content/models"
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


MODEL_PATH = (
    MODELS_DIR
    / "model-q4_K.gguf"
)


if not MODEL_PATH.exists():
    downloaded_path = hf_hub_download(
        repo_id=(
            "IlyaGusev/"
            "saiga_mistral_7b_gguf"
        ),
        filename="model-q4_K.gguf",
        local_dir=MODELS_DIR,
        token=False,
    )

    MODEL_PATH = Path(
        downloaded_path
    )


if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        "GGUF-модель не найдена."
    )


print(
    "Модель:",
    MODEL_PATH,
)

print(
    "Размер:",
    round(
        MODEL_PATH.stat().st_size
        / 1024**3,
        2,
    ),
    "GB",
)

model-q4_K.gguf: reconstructing file:   0%|          |  0.00B / 4.37GB            

model-q4_K.gguf: downloading bytes:           |  0.00B            

Модель: /content/models/model-q4_K.gguf
Размер: 4.07 GB


In [ ]:
from pathlib import Path
import gc

import torch

from rag_labor_code.config import AppConfig
from rag_labor_code.bootstrap import (
    build_rag_pipeline,
)


gc.collect()
torch.cuda.empty_cache()


config = AppConfig(
    saiga_model_path=MODEL_PATH,

    pdf_path=Path(
        "/content/"
        "RAG-Labor-Code-RU/"
        "data/raw/labor_code_rf.pdf"
    ),

    vector_index_dir=Path(
        "/content/"
        "RAG-Labor-Code-RU/"
        "data/processed/vector_index"
    ),

    nemo_config_dir=Path(
        "/content/"
        "RAG-Labor-Code-RU/"
        "configs/nemo"
    ),

    embedding_device="cuda",
    reranker_device="cuda",

    n_ctx=4096,
    n_gpu_layers=24,

    chat_format="chatml",

    rebuild_index=False,
)


pipeline = build_rag_pipeline(
    config
)


print("RAG pipeline полностью собран.")

In [ ]:
from time import perf_counter

from rag_labor_code.embeddings.model_factory import create_e5_embed_model
from rag_labor_code.bootstrap import load_or_build_vector_index
from rag_labor_code.retrieval.bm25_retriever import build_bm25_retriever
from rag_labor_code.reranking.cross_encoder import create_cross_encoder
from rag_labor_code.generation.saiga_generator import create_saiga_llm
from rag_labor_code.guardrails.nemo_guardrails import create_nemo_guardrails_adapter
from rag_labor_code.pipeline.rag_pipeline import RAGPipeline


def stage(name, func):
    print(f"\n>>> {name}...")
    start = perf_counter()

    result = func()

    print(
        f"<<< {name} готово за "
        f"{perf_counter() - start:.1f} сек."
    )

    return result


embed_model = stage(
    "1. Загрузка E5",
    lambda: create_e5_embed_model(
        device=config.embedding_device,
    ),
)

index, nodes = stage(
    "2. PDF + nodes + embeddings + vector index",
    lambda: load_or_build_vector_index(
        config=config,
        embed_model=embed_model,
    ),
)

bm25 = stage(
    "3. BM25",
    lambda: build_bm25_retriever(
        nodes=nodes,
        top_k=config.pipeline_config.retrieval_top_k,
    ),
)

reranker = stage(
    "4. CrossEncoder",
    lambda: create_cross_encoder(
        device=config.reranker_device,
    ),
)

llm = stage(
    "5. Saiga",
    lambda: create_saiga_llm(
        model_path=config.saiga_model_path,
        n_ctx=config.n_ctx,
        n_gpu_layers=config.n_gpu_layers,
        n_threads=config.n_threads,
        chat_format=config.chat_format,
    ),
)

nemo = stage(
    "6. NeMo Guardrails",
    lambda: create_nemo_guardrails_adapter(
        config_path=config.nemo_config_dir,
        llm=llm,
    ),
)

pipeline = RAGPipeline(
    index=index,
    bm25_retriever=bm25,
    reranker=reranker,
    llm=llm,
    config=config.pipeline_config,
    nemo_guardrails=nemo,
)

print("\n✅ RAG pipeline полностью собран.")

In [ ]:
print("Kernel работает")

In [ ]:
question = (
    "Какова нормальная "
    "продолжительность рабочего "
    "времени в неделю?"
)


result = pipeline.answer(
    question
)


print("=== РЕЗУЛЬТАТ ===")
print()

print(
    "Blocked:",
    result.blocked,
)

print(
    "Reason:",
    result.reason,
)

print()
print("Ответ:")
print(result.answer)

print()
print("Источники:")


for number, source in enumerate(
    result.sources,
    start=1,
):
    print()
    print(
        f"{number}. "
        f"Статья {source.article_num} "
        f"— {source.title}"
    )

    print(
        "Источник:",
        source.source,
    )

    print(
        "Score:",
        round(source.score, 4),
    )

In [ ]:
!nvidia-smi

In [ ]:
from rag_labor_code.ui.gradio_app import (
    launch_gradio_app,
)


launch_gradio_app(
    pipeline=pipeline,
    server_name="0.0.0.0",
    server_port=7860,
    share=True,
)